# Gaussian Splatting: Generating 3D Models from Images

**Author:** Carlos Melo — [sigmoidal.ai](https://sigmoidal.ai)

**Modified by:** Rafael Gomes da Silva — [GitHub](https://github.com/rafaelgscc)


Test notebook for generating a complete 3D model from a single 2D image using **Gaussian Splatting**.

To do this, we will use the **LGM (Large Gaussian Model)**, which combines two models:
1. **Multi-view Diffusion**: Generates 4 views of the object from a single photo.
2. **Large Gaussian Model**: Converts these 4 views into a Gaussian Splatting model.

## Setup

Install the LGM and visualization dependencies. The Gaussian rasterizer (`diff-gaussian-rasterization`) is compiled from source.

In [ ]:
!pip install -q diffusers accelerate transformers xformers kiui plyfile plotly requests matplotlib pillow
!pip install -U mslk --index-url https://download.pytorch.org/whl/cu130

# Compile diff_gaussian_rasterization from source
!pip install --no-build-isolation git+https://github.com/ashawkey/diff-gaussian-rasterization.git

In [ ]:
# Environment check, core imports, and global configuration
import gc
import os
import sys
import numpy as np
import torch

# Fixed random seed for reproducibility across diffusion sampling and 3D point subsampling
# (42 is a common ML convention; change this integer to generate different variations)
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

if not torch.cuda.is_available():
    raise RuntimeError(
        "CUDA GPU not detected. LGM and diff-gaussian-rasterization require a CUDA-enabled GPU."
    )

DEVICE = "cuda"
DTYPE = torch.float16

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"CUDA Available: True")
print(f"GPU: {gpu_name}")
print(f"VRAM: {vram_gb:.1f} GB")

if vram_gb < 10.0:
    print("Warning: Less than 10 GB of VRAM detected. Models will be offloaded between steps to reduce OOM risk.")

## Loading Models

Load the two pipeline models. To optimize GPU VRAM usage, models are moved to CUDA only when needed during inference.

In [ ]:
# Workaround for kiui typing incompatibilities in Python 3.13+
if sys.version_info >= (3, 13):
    import builtins
    from typing import Literal, Optional, Sequence, Union

    builtins.Union = Union
    builtins.Sequence = Sequence
    builtins.Optional = Optional
    builtins.Literal = Literal
    builtins.Tensor = torch.Tensor
    builtins.ndarray = np.ndarray

from diffusers import DiffusionPipeline

def load_pipeline(model_id: str, device: str = "cpu") -> DiffusionPipeline:
    """Load a custom Hugging Face DiffusionPipeline with clear error handling."""
    try:
        pipe = DiffusionPipeline.from_pretrained(
            model_id,
            custom_pipeline=model_id,
            torch_dtype=DTYPE,
            trust_remote_code=True,
        )
        if device and device != "cpu":
            pipe = pipe.to(device)
        return pipe
    except torch.cuda.OutOfMemoryError as exc:
        torch.cuda.empty_cache()
        raise RuntimeError(
            f"Out of GPU memory while loading '{model_id}'. Try freeing VRAM or restarting the runtime."
        ) from exc
    except Exception as exc:
        raise RuntimeError(f"Failed to load pipeline '{model_id}': {exc}") from exc

# Model 1: Multi-view diffusion (generates 4 views from 1 image)
image_pipeline = load_pipeline("dylanebert/multi-view-diffusion", device=DEVICE)
print("Multi-view diffusion loaded.")

In [ ]:
# Model 2: Large Gaussian Model (converts the 4 views into 3D Gaussians)
# Kept on CPU until Step 2 to avoid holding both models in GPU VRAM simultaneously
splat_pipeline = load_pipeline("dylanebert/LGM", device="cpu")
print("LGM loaded (staged on CPU until Step 2).")

## Loading an Input Image

Using an example image. You can replace `image_source` with any URL or local file path of an object against a clean background.

In [ ]:
from io import BytesIO
import matplotlib.pyplot as plt
import requests
from PIL import Image, UnidentifiedImageError

def load_input_image(source: str, timeout: int = 15) -> Image.Image:
    """Load and preprocess an input image from a URL or local file path.

    Downloads the image (with timeout and HTTP status validation) if `source` is
    a URL, or opens it from disk if it is a local path. Images with transparency
    (RGBA, LA, or palette-based PNGs) are composited onto a solid white
    background before converting to 3-channel RGB as expected by the diffusion
    pipeline.

    Args:
        source: HTTP/HTTPS URL or local filesystem path to the image.
        timeout: Request timeout in seconds when downloading from a URL.

    Returns:
        A 3-channel RGB `PIL.Image.Image` ready for inference.

    Raises:
        FileNotFoundError: If a local path does not exist.
        RuntimeError: If downloading from a URL fails.
        ValueError: If the loaded file is not a valid image.
    """
    try:
        if source.startswith(("http://", "https://")):
            response = requests.get(source, timeout=timeout)
            response.raise_for_status()
            img = Image.open(BytesIO(response.content))
        else:
            if not os.path.exists(source):
                raise FileNotFoundError(f"Local image not found: '{source}'")
            img = Image.open(source)
    except requests.RequestException as exc:
        raise RuntimeError(f"Failed to download image from '{source}': {exc}") from exc
    except UnidentifiedImageError as exc:
        raise ValueError(f"The file at '{source}' is not a valid image.") from exc

    # Composite transparent PNGs onto a white background before converting to RGB
    if img.mode in ("RGBA", "LA") or (img.mode == "P" and "transparency" in img.info):
        img = img.convert("RGBA")
        background = Image.new("RGBA", img.size, (255, 255, 255, 255))
        img = Image.alpha_composite(background, img).convert("RGB")
    else:
        img = img.convert("RGB")

    return img

image_source = "https://huggingface.co/datasets/dylanebert/3d-arena/resolve/main/inputs/images/a_cat_statue.jpg"
image = load_input_image(image_source)

plt.figure(figsize=(5, 5))
plt.imshow(image)
plt.axis("off")
plt.title("Input Image")
plt.show()

### Step 1: Generating Multi-Views

The diffusion model imagines what the object looks like from 4 different angles (front, side, back, and the other side). This is achieved through multi-view diffusion, an extension of the diffusion models used to generate images.

In [ ]:
import xformers.ops

# Replace xformers attention with PyTorch's native scaled_dot_product_attention (SDPA)
def sdpa_memory_efficient_attention(q, k, v, attn_bias=None, op=None, scale=None, p=0.0):
    """Drop-in replacement for xformers.ops.memory_efficient_attention using PyTorch SDPA."""
    if q.ndim == 4:
        # xformers uses (B, L, H, D); PyTorch SDPA expects (B, H, L, D)
        q, k, v = q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2)
        out = torch.nn.functional.scaled_dot_product_attention(
            q, k, v, attn_mask=attn_bias, dropout_p=p, scale=scale
        )
        return out.transpose(1, 2)
    return torch.nn.functional.scaled_dot_product_attention(
        q, k, v, attn_mask=attn_bias, dropout_p=p, scale=scale
    )

xformers.ops.memory_efficient_attention = sdpa_memory_efficient_attention

input_image = np.array(image, dtype=np.float32) / 255.0

try:
    image_pipeline.to(DEVICE)
    with torch.inference_mode():
        multi_view_images = image_pipeline(
            "",
            input_image,
            guidance_scale=5,
            num_inference_steps=30,
            elevation=0,
        )
except torch.cuda.OutOfMemoryError as exc:
    torch.cuda.empty_cache()
    raise RuntimeError(
        "Out of GPU memory during multi-view generation. Try closing other GPU processes."
    ) from exc
finally:
    # Offload image_pipeline to CPU RAM to free VRAM for LGM
    image_pipeline.to("cpu", silence_dtype_warnings=True)
    gc.collect()
    torch.cuda.empty_cache()

if len(multi_view_images) == 0:
    raise RuntimeError("Multi-view pipeline returned no images.")

print(f"Generated {len(multi_view_images)} views")
print(f"Shape of each view: {multi_view_images[0].shape}")

The model returns the generated views of the object from different angles. Let's visualize the 4 generated views below.

In [ ]:
# Visualize the generated views
labels = ["Right", "Front", "Left", "Back"]
n_views = min(len(multi_view_images), len(labels))

fig, axes = plt.subplots(1, n_views, figsize=(4 * n_views, 4))
if n_views == 1:
    axes = [axes]

for i in range(n_views):
    axes[i].imshow(multi_view_images[i])
    axes[i].set_title(labels[i])
    axes[i].axis("off")

plt.suptitle("Views generated by the diffusion model", fontsize=14)
plt.tight_layout()
plt.show()

From a single photo, the diffusion model inferred what the object looks like when viewed from other angles. It is not perfect, but it is sufficient to generate a convincing 3D model.

### Step 2: Generating the 3D Model

Now, LGM takes the 4 views and generates a set of 3D Gaussians.

In [ ]:
try:
    splat_pipeline.to(DEVICE)
    with torch.inference_mode():
        splat = splat_pipeline(multi_view_images)
except torch.cuda.OutOfMemoryError as exc:
    torch.cuda.empty_cache()
    raise RuntimeError(
        "Out of GPU memory during 3D Gaussian generation."
    ) from exc
finally:
    torch.cuda.empty_cache()

print("3D model successfully generated!")
print(f"Type: {type(splat)}")
print(f"Shape: {splat.shape}")

### Saving the Model

We will save it as a `.ply` file, which can be viewed in any Gaussian Splatting viewer.

In [ ]:
output_path = "test_model_01.ply"

output_dir = os.path.dirname(output_path)
if output_dir:
    os.makedirs(output_dir, exist_ok=True)

try:
    splat_pipeline.save_ply(splat, output_path)
except Exception as exc:
    raise RuntimeError(f"Failed to save PLY file to '{output_path}': {exc}") from exc

if not os.path.exists(output_path) or os.path.getsize(output_path) == 0:
    raise RuntimeError(f"Output file '{output_path}' was not created or is empty.")

size_mb = os.path.getsize(output_path) / (1024 * 1024)
print(f"Model: {output_path}")
print(f"Size: {size_mb:.1f} MB")

## Inspecting the Generated Model

Now, let's open the generated `.ply` file and examine the Gaussians, just as we did in the previous notebook.

In [ ]:
from IPython.display import Markdown, display
from plyfile import PlyData

if not os.path.exists(output_path):
    raise FileNotFoundError(f"PLY file not found: '{output_path}'. Run the saving cell first.")

try:
    plydata = PlyData.read(output_path)
except Exception as exc:
    raise RuntimeError(f"Failed to read PLY file '{output_path}': {exc}") from exc

if "vertex" not in plydata:
    raise KeyError(f"PLY file '{output_path}' does not contain a 'vertex' element.")

vertex = plydata["vertex"]
prop_names = [p.name for p in vertex.properties]

print(f"Number of useful Gaussians saved: {len(vertex):,}")
print(f"Properties: {prop_names}")

total_generated = splat.shape[1] if "splat" in globals() and hasattr(splat, "shape") and len(splat.shape) >= 2 else 65536
splat_shape_str = str(tuple(splat.shape)) if "splat" in globals() and hasattr(splat, "shape") else "(1, 65536, 14)"
pruned_count = max(total_generated - len(vertex), 0)

display(Markdown(
    f"The model internally generates **{total_generated:,}** Gaussians (a tensor with shape `{splat_shape_str}`), "
    f"but when saving the `.ply` file, Gaussians with very low opacity are removed ({pruned_count:,} pruned). "
    f"As a result, the final file contains **{len(vertex):,}** useful Gaussians."
))

In [ ]:
import plotly.graph_objects as go

MAX_PLOT_POINTS = 20_000  # Subsample for smooth interactive 3D rendering

prop_name_set = set(prop_names)
required_coords = {"x", "y", "z"}
if not required_coords.issubset(prop_name_set):
    raise KeyError(f"Missing coordinate properties in PLY vertex data: {required_coords - prop_name_set}")

xyz = np.stack([vertex["x"], vertex["y"], vertex["z"]], axis=1)

# Extract RGB colors depending on the LGM PLY output format
if {"red", "green", "blue"}.issubset(prop_name_set):
    r = np.clip(np.asarray(vertex["red"], dtype=np.float32) / 255.0, 0.0, 1.0)
    g = np.clip(np.asarray(vertex["green"], dtype=np.float32) / 255.0, 0.0, 1.0)
    b = np.clip(np.asarray(vertex["blue"], dtype=np.float32) / 255.0, 0.0, 1.0)
elif {"f_dc_0", "f_dc_1", "f_dc_2"}.issubset(prop_name_set):
    C0 = 0.28209479177387814
    r = np.clip(0.5 + C0 * np.asarray(vertex["f_dc_0"], dtype=np.float32), 0.0, 1.0)
    g = np.clip(0.5 + C0 * np.asarray(vertex["f_dc_1"], dtype=np.float32), 0.0, 1.0)
    b = np.clip(0.5 + C0 * np.asarray(vertex["f_dc_2"], dtype=np.float32), 0.0, 1.0)
else:
    r = g = b = np.full(len(vertex), 0.5, dtype=np.float32)

rgb = (np.stack([r, g, b], axis=1) * 255).astype(np.uint8)

if len(xyz) > MAX_PLOT_POINTS:
    rng = np.random.default_rng(SEED)
    idx = rng.choice(len(xyz), size=MAX_PLOT_POINTS, replace=False)
    xyz_plot = xyz[idx]
    rgb_plot = rgb[idx]
else:
    xyz_plot = xyz
    rgb_plot = rgb

colors = [f"rgb({ri},{gi},{bi})" for ri, gi, bi in rgb_plot]
print(f"Prepared {len(xyz_plot):,} / {len(xyz):,} points for 3D visualization.")

In [ ]:
fig = go.Figure(data=[go.Scatter3d(
    x=xyz_plot[:, 0],
    y=xyz_plot[:, 1],
    z=xyz_plot[:, 2],
    mode="markers",
    marker=dict(size=1.5, color=colors, opacity=0.6),
)])
fig.update_layout(
    title=f"3D model generated from a single photo ({len(xyz_plot):,} Gaussians shown)",
    font=dict(color="white"),
    width=800,
    height=600,
    scene=dict(aspectmode="data", bgcolor="black"),
    paper_bgcolor="black",
)
fig.show()

## Viewing in the Browser

The Plotly visualization displays the centers of the Gaussians, but does not render them as overlapping ellipsoids (which is what produces the photorealistic result).

To see the actual rendered result, download the `.ply` file and drag it into the web viewer:

**[https://antimatter15.com/splat/](https://antimatter15.com/splat/)**

Or use SuperSplat to view and edit it:

**[https://superspl.at/](https://superspl.at/)**